In [ ]:
import os
import time
import random
import pandas as pd
import yfinance as yf
from tqdm import tqdm

INPUT_FILE = "EQUITY_L.csv"
OUTPUT_FILE = "nse_daily_prices_long_2021_2025.parquet"
CHECKPOINT_FILE = "download_checkpoint.txt"
TEMP_DIR = "temp_batches"

GLOBAL_START = pd.to_datetime("2021-01-01")
GLOBAL_END = pd.to_datetime("2025-12-31")
SAVE_EVERY = 25
MAX_RETRIES = 4
TIMEOUT = 2

os.makedirs(TEMP_DIR, exist_ok=True)

equity_df = pd.read_csv(INPUT_FILE)
equity_df.columns = equity_df.columns.str.strip()
equity_df["DATE OF LISTING"] = pd.to_datetime(equity_df["DATE OF LISTING"], errors="coerce", dayfirst=True)
equity_df = equity_df.dropna(subset=["SYMBOL"]).copy()
equity_df["SYMBOL"] = equity_df["SYMBOL"].astype(str)

symbols = equity_df["SYMBOL"].unique().tolist()
symbol_to_listing = (
    equity_df.drop_duplicates("SYMBOL")
    .set_index("SYMBOL")["DATE OF LISTING"]
    .to_dict()
)

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            txt = f.read().strip()
            return int(txt) if txt else 0
    return 0

def save_checkpoint(idx):
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(idx))

def load_done_symbols():
    done = set()
    if os.path.exists(TEMP_DIR):
        for fn in os.listdir(TEMP_DIR):
            if fn.endswith(".parquet"):
                try:
                    part = pd.read_parquet(os.path.join(TEMP_DIR, fn), columns=["symbol"])
                    done.update(part["symbol"].dropna().astype(str).unique())
                except Exception:
                    pass
    return done

def retry_info(ticker, max_retries=MAX_RETRIES, base_sleep=1):
    for attempt in range(max_retries):
        try:
            return yf.Ticker(ticker).info
        except Exception:
            time.sleep(base_sleep * (2 ** attempt) + random.uniform(0, 0.8))
    return {}

def retry_download(ticker, start_date, max_retries=MAX_RETRIES, base_sleep=1):
    for attempt in range(max_retries):
        try:
            return yf.download(
                tickers=ticker,
                start=start_date.strftime("%Y-%m-%d"),
                end="2026-01-01",
                interval="1d",
                auto_adjust=True,
                progress=False,
                threads=True,
                timeout=TIMEOUT
            )
        except Exception:
            time.sleep(base_sleep * (2 ** attempt) + random.uniform(0, 0.8))
    return pd.DataFrame()

start_idx = load_checkpoint()
done_symbols = load_done_symbols()
buffer_rows = []
batch_no = 1 + len([f for f in os.listdir(TEMP_DIR) if f.endswith(".parquet")])

for idx in tqdm(range(start_idx, len(symbols))):
    sym = symbols[idx]
    ticker = f"{sym}.NS"

    if sym in done_symbols:
        save_checkpoint(idx + 1)
        continue

    try:
        listing_date = symbol_to_listing.get(sym, pd.NaT)
        download_start = GLOBAL_START if pd.isna(listing_date) else max(listing_date, GLOBAL_START)

        info = retry_info(ticker)
        sector = info.get("sector", "Unknown")

        df = retry_download(ticker, download_start)

        if not df.empty:
            if isinstance(df.columns, pd.MultiIndex):
                price_cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in df.columns.get_level_values(0)]
                if price_cols:
                    sub = pd.concat([df[c] for c in price_cols], axis=1)
                    sub.columns = price_cols
                else:
                    sub = pd.DataFrame(index=df.index)
            else:
                sub = df[[c for c in ["Open", "High", "Low", "Close", "Volume"] if c in df.columns]].copy()

            sub = sub.reset_index()
            date_col = "Date" if "Date" in sub.columns else sub.columns[0]
            sub["date"] = pd.to_datetime(sub[date_col])
            sub["symbol"] = sym
            sub["sector"] = sector
            sub["start_date"] = download_start

            keep_cols = ["symbol", "sector", "start_date", "date"] + [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in sub.columns]
            sub = sub[keep_cols]
            sub = sub[(sub["date"] >= GLOBAL_START) & (sub["date"] <= GLOBAL_END)]

            buffer_rows.append(sub)

        time.sleep(random.uniform(0.05, 0.12))

    except Exception as e:
        print(f"Failed for {ticker}: {e}")
        time.sleep(0.5)

    if (idx + 1) % SAVE_EVERY == 0 or (idx + 1) == len(symbols):
        if buffer_rows:
            batch_df = pd.concat(buffer_rows, ignore_index=True)
            temp_file = os.path.join(TEMP_DIR, f"batch_{batch_no:05d}.parquet")
            batch_df.to_parquet(temp_file, index=False)
            print(f"Saved {temp_file} rows={len(batch_df)}")
            batch_no += 1
            buffer_rows = []

        save_checkpoint(idx + 1)
        print(f"Saved progress at symbol {idx + 1}/{len(symbols)}")

if buffer_rows:
    batch_df = pd.concat(buffer_rows, ignore_index=True)
    temp_file = os.path.join(TEMP_DIR, f"batch_{batch_no:05d}.parquet")
    batch_df.to_parquet(temp_file, index=False)
    print(f"Saved {temp_file} rows={len(batch_df)}")

all_parts = []
for fn in sorted(os.listdir(TEMP_DIR)):
    if fn.endswith(".parquet"):
        all_parts.append(pd.read_parquet(os.path.join(TEMP_DIR, fn)))

final_df = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame()
final_df.to_parquet(OUTPUT_FILE, index=False)

save_checkpoint(len(symbols))
print(f"Done. Output saved to {OUTPUT_FILE}")
print(f"Temporary batch files saved in: {TEMP_DIR}")

/tmp/ipykernel_27140/1138167869.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  equity_df["DATE OF LISTING"] = pd.to_datetime(equity_df["DATE OF LISTING"], errors="coerce", dayfirst=True)
  0%|          | 3/2367 [00:02<26:49,  1.47it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['3BBLACKBIO.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-04-20 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1776623400, endDate = 1767205800")')
  1%|          | 14/2367 [00:06<16:21,  2.40it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AARNAV.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-02-25 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1771957800, endDate = 1767205800")')
  1%|          |

Saved temp_batches/batch_00001.parquet rows=24407
Saved progress at symbol 25/2367


  1%|▏         | 34/2367 [00:15<16:10,  2.40it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ABMKNO.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-04-20 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1776623400, endDate = 1767205800")')
  2%|▏         | 45/2367 [00:23<23:14,  1.67it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ACSTECH.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-04-20 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1776623400, endDate = 1767205800")')
  2%|▏         | 50/2367 [00:25<18:40,  2.07it/s]

Saved temp_batches/batch_00002.parquet rows=19779
Saved progress at symbol 50/2367


  2%|▏         | 57/2367 [00:28<17:39,  2.18it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ADVAIT.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-01-20 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1768847400, endDate = 1767205800")')
  3%|▎         | 64/2367 [00:31<15:05,  2.54it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AEPL.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-03-12 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1773253800, endDate = 1767205800")')
  3%|▎         | 75/2367 [00:35<15:46,  2.42it/s]

...

 98%|█████████▊| 2327/2367 [26:40<00:17,  2.28it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WIMPLAST.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-04-20 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1776623400, endDate = 1767205800")')
 99%|█████████▊| 2336/2367 [26:44<00:14,  2.17it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WPIL.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-04-20 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1776623400, endDate = 1767205800")')
 99%|█████████▉| 2350/2367 [28:14<00:35,  2.09s/it]

Saved temp_batches/batch_00094.parquet rows=24537
Saved progress at symbol 2350/2367


100%|█████████▉| 2357/2367 [28:17<00:05,  1.74it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ZFSTEERING.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-04-20 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1776623400, endDate = 1767205800")')
100%|█████████▉| 2362/2367 [28:20<00:02,  2.08it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ZSARACOM.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-04-20 -> 2026-01-01) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1776623400, endDate = 1767205800")')
100%|██████████| 2367/2367 [28:22<00:00,  1.39it/s]

Saved temp_batches/batch_00095.parquet rows=17815
Saved progress at symbol 2367/2367


Done. Output saved to nse_daily_prices_long_2021_2025.parquet
Temporary batch files saved in: temp_batches


preprocessing

In [ ]:
!pip install pandas_market_calendars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: toolz
    Found existing installation: toolz 0.12.1
    Uninstalling toolz-0.12.1:
      Successfully uninstalled toolz-0.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.


In [ ]:
import pandas as pd
import numpy as np
import pandas_market_calendars as mcal
import yfinance as yf
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import MinMaxScaler

INPUT_FILE = "nse_daily_prices_long_2021_2025.parquet"
OUTPUT_FILE = "nse_part2_features_masked.parquet"
REPORT_FILE = "part2_preprocessing_report.csv"

GLOBAL_START = pd.Timestamp("2021-01-01")
GLOBAL_END = pd.Timestamp("2025-12-31")
MASK_THRESHOLD = 5
SENTINEL = -99.0
# Removed MAX_STOCKS = 5 to process all stocks

df = pd.read_parquet(INPUT_FILE)
df["date"] = pd.to_datetime(df["date"])
df["start_date"] = pd.to_datetime(df["start_date"])

for c in ["Open", "High", "Low", "Close", "Volume"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Modified to process all unique symbols
symbols = df["symbol"].dropna().drop_duplicates().tolist()
df = df[df["symbol"].isin(symbols)].copy()

nse = mcal.get_calendar("NSE")
schedule = nse.schedule(start_date=GLOBAL_START, end_date=GLOBAL_END)
trading_days = pd.DatetimeIndex(schedule.index)

bench = yf.download(
    "^NSEI",
    start="2021-01-01",
    end="2026-01-01",
    interval="1d",
    auto_adjust=True,
    progress=False,
    threads=True,
    timeout=20
)

bench = bench.reset_index()
bench_date_col = "Date" if "Date" in bench.columns else bench.columns[0]
bench["date"] = pd.to_datetime(bench[bench_date_col])
bench = bench[["date", "Close"]].rename(columns={"Close": "nifty_close"})
bench = bench.set_index("date").reindex(trading_days)
bench["nifty_close"] = bench["nifty_close"].ffill()
bench["nifty_return"] = np.log(bench["nifty_close"] / bench["nifty_close"].shift(1))
bench["nifty_return"] = bench["nifty_return"].ffill()

def find_blocks(mask):
    blocks = []
    i = 0
    n = len(mask)
    while i < n:
        if mask[i]:
            j = i
            while j < n and mask[j]:
                j += 1
            blocks.append((i, j - 1))
            i = j
        else:
            i += 1
    return blocks

def rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

def atr(df_):
    high_low = df_["High"] - df_["Low"]
    high_close = (df_["High"] - df_["Close"].shift()).abs()
    low_close = (df_["Low"] - df_["Close"].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(14).mean()

def mfi(df_, window=14):
    tp = (df_["High"] + df_["Low"] + df_["Close"]) / 3
    mf = tp * df_["Volume"]
    delta_tp = tp.diff()
    pos = mf.where(delta_tp > 0, 0.0)
    neg = mf.where(delta_tp < 0, 0.0).abs()
    pos_sum = pos.rolling(window).sum()
    neg_sum = neg.rolling(window).sum()
    mfr = pos_sum / neg_sum
    return 100 - (100 / (1 + mfr))

parts = []
report_rows = []

for sym in symbols:
    g = df[df["symbol"] == sym].sort_values("date").copy()
    start_date = g["start_date"].iloc[0]
    if pd.isna(start_date):
        start_date = g["date"].min()

    cal = trading_days[(trading_days >= start_date) & (trading_days <= GLOBAL_END)]
    g = g.set_index("date").reindex(cal)
    g.index.name = "date"

    g["symbol"] = sym
    if "sector" in g.columns:
        g["sector"] = g["sector"].ffill()
    g["start_date"] = start_date

    rows_before = len(g)
    close_missing_mask = g["Close"].isna().to_numpy() if "Close" in g.columns else np.zeros(len(g), dtype=bool)
    blocks = find_blocks(close_missing_mask)
    missing_close_before = int(close_missing_mask.sum())
    missing_pct_before = round(100 * missing_close_before / rows_before, 2) if rows_before else 0

    g["is_missing"] = 0
    filled_count = 0
    masked_count = 0

    for start, length in blocks:
        if length >= MASK_THRESHOLD:
            g.iloc[start:start + length, g.columns.get_loc("is_missing")] = 1
            masked_count += length
        else:
            if start > 0 and pd.notna(g["Close"].iloc[start - 1]):
                g.iloc[start:start + length, g.columns.get_loc("Close")] = g["Close"].iloc[start - 1]
                filled_count += length

    g["Close"] = g["Close"].ffill()
    for c in ["Open", "High", "Low"]:
        if c in g.columns:
            g[c] = g[c].fillna(g["Close"])
    if "Volume" in g.columns:
        g["Volume"] = g["Volume"].fillna(0)

    g = g[g["Close"].notna()].copy()
    if g.empty:
        report_rows.append({
            "symbol": sym,
            "status": "dropped",
            "reason": "no valid close after alignment",
            "start_date": str(start_date.date()) if pd.notna(start_date) else "",
            "rows_before": rows_before,
            "rows_after": 0,
            "missing_close_before_fill": missing_close_before,
            "missing_pct_before_fill": missing_pct_before,
            "filled_count": filled_count,
            "masked_count": masked_count,
            "remaining_na": 0,
            "warmup_rows_dropped": 0,
            "adf_pvalue": np.nan,
            "stationary_flag": False,
            "used_scaler": "",
            "notes": "no valid close after fill"
        })
        continue

    g["day_of_week"] = g.index.dayofweek
    g["month"] = g.index.month
    g["log_return"] = np.log(g["Close"] / g["Close"].shift(1))
    g["dow_sin"] = np.sin(2 * np.pi * g["day_of_week"] / 7.0)
    g["dow_cos"] = np.cos(2 * np.pi * g["day_of_week"] / 7.0)
    g["month_sin"] = np.sin(2 * np.pi * g["month"] / 12.0)
    g["month_cos"] = np.cos(2 * np.pi * g["month"] / 12.0)

    g["rsi_14"] = rsi(g["Close"], 14)
    g["ema_12"] = ema(g["Close"], 12)
    g["ema_26"] = ema(g["Close"], 26)
    g["macd"] = g["ema_12"] - g["ema_26"]
    g["macd_signal"] = ema(g["macd"], 9)

    g["sma_20"] = g["Close"].rolling(20).mean()
    g["sma_50"] = g["Close"].rolling(50).mean()
    g["dist_sma_20"] = (g["Close"] / g["sma_20"]) - 1
    g["dist_sma_50"] = (g["Close"] / g["sma_50"]) - 1

    for n in [3, 5, 10]:
        g[f"roc_{n}"] = g["Close"].pct_change(n)

    bb_mid = g["Close"].rolling(20).mean()
    bb_std = g["Close"].rolling(20).std()
    g["bb_width"] = ((bb_mid + 2 * bb_std) - (bb_mid - 2 * bb_std)) / bb_mid

    g["atr_14"] = atr(g)
    g["atr_pct"] = g["atr_14"] / g["Close"]

    g["hist_vol_10"] = g["log_return"].rolling(10).std()
    g["hist_vol_21"] = g["log_return"].rolling(21).std()

    vol_mean = g["Volume"].rolling(20).mean()
    vol_std = g["Volume"].rolling(20).std()
    g["vol_zscore"] = (g["Volume"] - vol_mean) / vol_std

    obv = [0.0]
    for i in range(1, len(g)):
        if g["Close"].iloc[i] > g["Close"].iloc[i - 1]:
            obv.append(obv[-1] + g["Volume"].iloc[i])
        elif g["Close"].iloc[i] < g["Close"].iloc[i - 1]:
            obv.append(obv[-1] - g["Volume"].iloc[i])
        else:
            obv.append(obv[-1])
    g["obv"] = obv

    g["mfi_14"] = mfi(g, 14)

    rng = (g["High"] - g["Low"]).replace(0, np.nan)
    g["body_size"] = (g["Close"] - g["Open"]) / rng
    g["upper_shadow_ratio"] = (g["High"] - g[["Open", "Close"]].max(axis=1)) / rng
    g["lower_shadow_ratio"] = (g[["Open", "Close"]].min(axis=1) - g["Low"]) / rng

    bench_aligned = bench.reindex(g.index)
    g["nifty_close"] = bench_aligned["nifty_close"].values
    g["nifty_return"] = bench_aligned["nifty_return"].values
    g["rel_strength"] = g["Close"] / g["nifty_close"]
    g["bench_corr_20"] = g["log_return"].rolling(20).corr(g["nifty_return"])

    try:
        pval = adfuller(g["log_return"].dropna())[1]
    except Exception:
        pval = np.nan

    feature_cols = [
        "Open", "High", "Low", "Close", "Volume",
        "log_return", "rsi_14", "ema_12", "ema_26", "macd", "macd_signal",
        "sma_20", "sma_50", "dist_sma_20", "dist_sma_50",
        "roc_3", "roc_5", "roc_10", "bb_width", "atr_14", "atr_pct",
        "hist_vol_10", "hist_vol_21", "vol_zscore", "obv", "mfi_14",
        "body_size", "upper_shadow_ratio", "lower_shadow_ratio",
        "dow_sin", "dow_cos", "month_sin", "month_cos",
        "bench_corr_20", "rel_strength"
    ]

    for c in feature_cols:
        if c in g.columns:
            g[c] = g[c].replace([np.inf, -np.inf], np.nan)

    warmup_mask = g[feature_cols].isna().any(axis=1)
    warmup_rows = int(warmup_mask.sum())

    g_feat = g.loc[~warmup_mask].copy()
    if g_feat.empty:
        report_rows.append({
            "symbol": sym,
            "status": "dropped",
            "reason": "all rows lost in warmup",
            "start_date": str(start_date.date()) if pd.notna(start_date) else "",
            "rows_before": rows_before,
            "rows_after": 0,
            "missing_close_before_fill": missing_close_before,
            "missing_pct_before_fill": missing_pct_before,
            "filled_count": filled_count,
            "masked_count": masked_count,
            "remaining_na": 0,
            "warmup_rows_dropped": warmup_rows,
            "adf_pvalue": pval,
            "stationary_flag": pd.notna(pval) and pval <= 0.05,
            "used_scaler": "",
            "notes": "warmup removed all rows"
        })
        continue

    scaler = MinMaxScaler(feature_range=(-1, 1))
    numeric_cols = [c for c in feature_cols if c in g_feat.columns]
    g_feat[numeric_cols] = scaler.fit_transform(g_feat[numeric_cols])

    for c in numeric_cols:
        g_feat.loc[g_feat["is_missing"] == 1, c] = SENTINEL

    g_feat["adf_pvalue"] = pval
    g_feat["stationary_flag"] = pd.notna(pval) and pval <= 0.05

    # ── FIX: reset index FIRST so "date" becomes a regular column ────────
    g_feat = g_feat.reset_index()   # index named "date" → column "date"

    out_cols = [
        "symbol", "sector", "start_date", "date", "is_missing",
        "Open", "High", "Low", "Close", "Volume",
        "log_return", "rsi_14", "ema_12", "ema_26", "macd", "macd_signal",
        "sma_20", "sma_50", "dist_sma_20", "dist_sma_50",
        "roc_3", "roc_5", "roc_10", "bb_width", "atr_14", "atr_pct",
        "hist_vol_10", "hist_vol_21", "vol_zscore", "obv", "mfi_14",
        "body_size", "upper_shadow_ratio", "lower_shadow_ratio",
        "dow_sin", "dow_cos", "month_sin", "month_cos",
        "bench_corr_20", "rel_strength",
        "adf_pvalue", "stationary_flag"
    ]
    out_cols = [c for c in out_cols if c in g_feat.columns]

    parts.append(g_feat[out_cols])   # no second reset_index() needed

    report_rows.append({
        "symbol": sym,
        "status": "kept",
        "reason": "",
        "start_date": str(start_date.date()) if pd.notna(start_date) else "",
        "rows_before": rows_before,
        "rows_after": len(g_feat),
        "missing_close_before_fill": missing_close_before,
        "missing_pct_before_fill": missing_pct_before,
        "filled_count": filled_count,
        "masked_count": masked_count,
        "remaining_na": int(g_feat.isna().sum().sum()),
        "warmup_rows_dropped": warmup_rows,
        "adf_pvalue": pval,
        "stationary_flag": pd.notna(pval) and pval <= 0.05,
        "used_scaler": "MinMaxScaler(-1,1)",
        "notes": f"sentinel={SENTINEL}; mask_threshold={MASK_THRESHOLD}; fill_then_mask"
    })

result_df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
report_df = pd.DataFrame(report_rows)

result_df.to_parquet(OUTPUT_FILE, index=False)
report_df.to_csv(REPORT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Saved:", REPORT_FILE)
print("Columns:", result_df.columns.tolist())
print(result_df.head())

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2


Saved: nse_part2_features_masked.parquet
Saved: part2_preprocessing_report.csv
Columns: ['symbol', 'sector', 'start_date', 'date', 'is_missing', 'Open', 'High', 'Low', 'Close', 'Volume', 'log_return', 'rsi_14', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'sma_20', 'sma_50', 'dist_sma_20', 'dist_sma_50', 'roc_3', 'roc_5', 'roc_10', 'bb_width', 'atr_14', 'atr_pct', 'hist_vol_10', 'hist_vol_21', 'vol_zscore', 'obv', 'mfi_14', 'body_size', 'upper_shadow_ratio', 'lower_shadow_ratio', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'bench_corr_20', 'rel_strength', 'adf_pvalue', 'stationary_flag']
      symbol           sector start_date       date  is_missing      Open  \
0  20MICRONS  Basic Materials 2021-01-01 2021-03-15           0 -0.973470   
1  20MICRONS  Basic Materials 2021-01-01 2021-03-16           0 -0.975735   
2  20MICRONS  Basic Materials 2021-01-01 2021-03-17           0 -0.966676   
3  20MICRONS  Basic Materials 2021-01-01 2021-03-18           0 -0.972499   
4  20MICRONS  Basi

In [ ]:
df = pd.read_parquet(INPUT_FILE)
print(df.columns.tolist())
print(df.shape)
print(df.head(2))

['symbol', 'sector', 'start_date', 'date', 'is_missing', 'Open', 'High', 'Low', 'Close', 'Volume', 'log_return', 'rsi_14', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'sma_20', 'sma_50', 'dist_sma_20', 'dist_sma_50', 'roc_3', 'roc_5', 'roc_10', 'bb_width', 'atr_14', 'atr_pct', 'hist_vol_10', 'hist_vol_21', 'vol_zscore', 'obv', 'mfi_14', 'body_size', 'upper_shadow_ratio', 'lower_shadow_ratio', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'bench_corr_20', 'rel_strength', 'adf_pvalue', 'stationary_flag']
(2029014, 42)
      symbol           sector start_date       date  is_missing      Open  \
0  20MICRONS  Basic Materials 2021-01-01 2021-03-15           0 -0.973470   
1  20MICRONS  Basic Materials 2021-01-01 2021-03-16           0 -0.975735   

       High       Low     Close    Volume  ...  upper_shadow_ratio  \
0 -0.978566 -0.941359 -0.977044 -0.990037  ...           -0.500003   
1 -0.976990 -0.939053 -0.971224 -0.987401  ...           -0.461540   

   lower_shadow_ratio   dow_sin   

Reducing correlated stocks

In [ ]:
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
import matplotlib.pyplot as plt

INPUT_FILE = "nse_part2_features_masked.parquet"
OUTPUT_FILE = "nse_part2_features_masked_clustered.parquet"
REPORT_FILE = "nse_cluster_filter_report.csv"
SUMMARY_FILE = "nse_cluster_filter_summary.csv"
HEATMAP_FILE = "nse_corr_heatmap.png"

THRESHOLD = 0.95
LINKAGE_METHOD = "complete"

df = pd.read_parquet(INPUT_FILE)
df["date"] = pd.to_datetime(df["date"])
df = df.loc[:, ~df.columns.duplicated()].copy()

ret = df.pivot_table(index="date", columns="symbol", values="log_return", aggfunc="first").sort_index()
corr = ret.corr(method="pearson")

corr = corr.replace([np.inf, -np.inf], np.nan).fillna(0)
corr = corr.clip(lower=-1, upper=1)

dist = 1 - corr.abs()
dist = dist.replace([np.inf, -np.inf], np.nan).fillna(1)
dist = dist.clip(lower=0, upper=1)
np.fill_diagonal(dist.values, 0)

condensed = squareform(dist.values, checks=False)

if not np.isfinite(condensed).all():
    raise ValueError("Condensed distance matrix still contains non-finite values.")

Z = linkage(condensed, method=LINKAGE_METHOD)

cluster_ids = fcluster(Z, t=1 - THRESHOLD, criterion="distance")
cluster_map = pd.Series(cluster_ids, index=corr.columns, name="cluster_id")

meta = (
    df.groupby("symbol")
      .agg(
          avg_volume=("Volume", "mean"),
          n_rows=("symbol", "size"),
          first_date=("date", "min"),
          last_date=("date", "max")
      )
)

selected = []
cluster_rows = []

for cid in sorted(cluster_map.unique()):
    members = cluster_map[cluster_map == cid].index.tolist()
    submeta = meta.loc[members].copy()

    rep = submeta.sort_values(
        ["avg_volume", "n_rows", "last_date"],
        ascending=[False, False, False]
    ).index[0]

    selected.append(rep)

    for sym in members:
        cluster_rows.append({
            "symbol": sym,
            "cluster_id": int(cid),
            "representative": sym == rep,
            "avg_volume": float(meta.loc[sym, "avg_volume"]) if pd.notna(meta.loc[sym, "avg_volume"]) else np.nan,
            "n_rows": int(meta.loc[sym, "n_rows"]),
            "first_date": meta.loc[sym, "first_date"],
            "last_date": meta.loc[sym, "last_date"],
        })

reduced_df = df[df["symbol"].isin(selected)].copy()
reduced_df.to_parquet(OUTPUT_FILE, index=False)

report_df = pd.DataFrame(cluster_rows).sort_values(["cluster_id", "representative"], ascending=[True, False])
report_df.to_csv(REPORT_FILE, index=False)

summary = pd.DataFrame({
    "metric": ["original_symbols", "clusters", "selected_symbols", "original_rows", "reduced_rows", "threshold"],
    "value": [df["symbol"].nunique(), len(cluster_map.unique()), len(selected), len(df), len(reduced_df), THRESHOLD]
})
summary.to_csv(SUMMARY_FILE, index=False)

# plain heatmap of correlation matrix
plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, xticklabels=False, yticklabels=False)
plt.title("Stock Return Correlation Heatmap")
plt.tight_layout()
plt.savefig(HEATMAP_FILE, dpi=200, bbox_inches="tight")
plt.close()

print("Selected symbols:", selected)
print("Saved:", OUTPUT_FILE)
print("Saved:", REPORT_FILE)
print("Saved:", SUMMARY_FILE)
print("Saved:", HEATMAP_FILE)
print(summary)

Selected symbols: ['RSSOFTWARE', 'ETERNAL', 'JAIPURKURT', 'SERVOTECH', 'ZODIAC', 'SEYAIND', 'VISASTEEL', 'SUMEETINDS', 'SUPREMEENG', 'VIJIFIN', 'SRD', 'DIFFNKG', 'XTGLOBAL', 'ARKADE', 'SURAJLTD', 'ECOSMOBLTY', 'EUREKAFORB', 'CRAMC', 'BLUEJET', 'HONASA', 'ASKAUTOLTD', 'FINKURVE', 'HGM', 'TRANSRAILL', 'JKIPL', 'NIBE', 'KMEW', 'RUBICON', 'RHETAN', 'SAATVIKGL', 'SECMARK', 'EPACKPEB', 'AEGISVOPAK', 'TRAVELFOOD', 'ODIGMA', 'UMIYA-MRO', 'LAXMIDENTL', 'PATELRMART', 'HDBFS', 'AJAXENGG', 'GVPIL', 'ALLTIME', 'WEWORK', 'QPOWER', 'CHEMBONDCH', 'ACMESOLAR', 'REGAAL', 'KPEL', 'VMSTMT', 'IGCL', 'MOBIKWIK', 'MOSCHIP', 'KALPATARU', 'MWL', 'MCLOUD', 'ARSSBL', 'NATCAPSUQ', 'ALIVUS', 'CIFL', 'WAAREEINDO', 'NILASPACES', 'MEIL', 'WAAREERTL', 'SHANTIGOLD', 'SANATHAN', 'TIGERLOGS', 'SUNDROP', 'EUROPRATIK', 'GKENERGY', 'BHARATSE', 'LGEINDIA', 'INNOVANA', 'BMWVENTLTD', 'ELLEN', 'WAAREEENER', 'AFCONS', 'SAILIFE', 'FISCHER', 'HBLENGINE', 'DAMCAPITAL', 'MBEL', 'LLOYDSENT', 'BELLACASA', 'CARRARO', 'IIFLCAPS', 'SENOR

part 3

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

INPUT_FILE = "nse_part2_features_masked_clustered.parquet"
OUTPUT_DIR = "part3_chunks"
LOOKBACK = 30
HORIZON = 5
TARGET_COL = "log_return"
MASK_COL = "is_missing"
MAX_STOCKS = None   # set to None later if you want full reduced universe

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_parquet(INPUT_FILE)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df = df.loc[:, ~df.columns.duplicated()].copy()

symbols = df["symbol"].dropna().drop_duplicates().tolist()
if MAX_STOCKS is not None:
    symbols = symbols[:MAX_STOCKS]

base_features = [
    "Open", "High", "Low", "Close", "Volume",
    "log_return", "rsi_14", "ema_12", "ema_26", "macd", "macd_signal",
    "sma_20", "sma_50", "dist_sma_20", "dist_sma_50",
    "roc_3", "roc_5", "roc_10", "bb_width", "atr_14", "atr_pct",
    "hist_vol_10", "hist_vol_21", "vol_zscore", "obv", "mfi_14",
    "body_size", "upper_shadow_ratio", "lower_shadow_ratio",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "bench_corr_20", "rel_strength"
]
base_features = sorted([c for c in base_features if c in df.columns])

print("Symbols used:", symbols)
print("Features used:", base_features)

for sym in symbols:
    cols_needed = ["symbol", "start_date", "date", MASK_COL, TARGET_COL] + base_features
    cols_needed = [c for c in cols_needed if c in df.columns]

    g = df.loc[df["symbol"] == sym, cols_needed].copy()
    g = g.sort_values("date").reset_index(drop=True)
    g = g.loc[:, ~g.columns.duplicated()].copy()

    if len(g) < LOOKBACK + HORIZON:
        print("Skipping", sym, "not enough rows")
        continue

    for c in base_features + [TARGET_COL]:
        g[c] = pd.to_numeric(g[c], errors="coerce")
    g[MASK_COL] = pd.to_numeric(g[MASK_COL], errors="coerce").fillna(0).astype(np.int8)

    X = g[base_features].to_numpy(dtype=np.float32, copy=False)
    y = g[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    m = g[MASK_COL].to_numpy(dtype=np.int8, copy=False)
    dates = g["date"].to_numpy()
    start_dates = g["start_date"].to_numpy()

    rows = []
    last_start = len(g) - LOOKBACK - HORIZON + 1
    for start_idx in range(last_start):
        end_idx = start_idx + LOOKBACK - 1
        target_start = end_idx + 1
        target_end = target_start + HORIZON

        y_vec = y[target_start:target_end]
        if len(y_vec) < HORIZON or np.isnan(y_vec).any():
            continue

        row = {
            "symbol": sym,
            "start_date": start_dates[start_idx],
            "window_start": dates[start_idx],
            "window_end": dates[end_idx],
            "target_start": dates[target_start],
            "target_end": dates[target_end - 1],
            "missing_count_in_window": int(m[start_idx:end_idx + 1].sum()),
            "has_masked_step": int(m[start_idx:end_idx + 1].sum() > 0),
        }

        xw = X[start_idx:end_idx + 1]
        for t in range(LOOKBACK):
            for j, col in enumerate(base_features):
                row[f"{col}_t{t+1}"] = xw[t, j]

        for h in range(HORIZON):
            row[f"y_t{h+1}"] = y_vec[h]

        rows.append(row)

    if rows:
        out = pd.DataFrame(rows)
        out.to_parquet(os.path.join(OUTPUT_DIR, f"{sym}_windows.parquet"), index=False)
        print("Saved:", sym, out.shape)
        del out

    del g, X, y, m, dates, start_dates, rows
    gc.collect()

print("Done. Saved chunk files to:", OUTPUT_DIR)

Symbols used: ['AADHARHFC', 'ABCOTS', 'ABINFRA', 'ACMESOLAR', 'ACUTAAS', 'ADVANCE', 'AEGISVOPAK', 'AEROENTER', 'AFCONS', 'AFFORDABLE', 'AGARWALEYE', 'AGIIL', 'AHCL', 'AJAXENGG', 'ALIVUS', 'ALLDIGI', 'ALLTIME', 'AMANTA', 'ANTELOPUS', 'ANTHEM', 'ANUHPHR', 'AQYLON', 'ARFIN', 'ARKADE', 'ARSSBL', 'ASKAUTOLTD', 'ATHERENERG', 'ATLANTAELE', 'AVANTEL', 'AVL', 'AWFIS', 'BANSALWIRE', 'BCG', 'BELLACASA', 'BELRISE', 'BHARATSE', 'BIRLANU', 'BLACKBUCK', 'BLUECOAST', 'BLUEJET', 'BLUSPRING', 'BMWVENTLTD', 'BORANA', 'CANHLIFE', 'CARRARO', 'CEMPRO', 'CEWATER', 'CHEMBONDCH', 'CIFL', 'COHANCE', 'COMSYN', 'CPCAP', 'CPEDU', 'CPPLUS', 'CRAMC', 'CRIZAC', 'DAMCAPITAL', 'DBEIL', 'DDEVPLSTIK', 'DENTA', 'DEVX', 'DIFFNKG', 'EASTSILK', 'EBGNG', 'ECOSMOBLTY', 'EFCIL', 'EIEL', 'ELLEN', 'EMBDL', 'EPACKPEB', 'ETERNAL', 'EUREKAFORB', 'EUROBOND', 'EUROPRATIK', 'FABTECH', 'FILATFASH', 'FINKURVE', 'FISCHER', 'FLAIR', 'GANESHCP', 'GANESHHOU', 'GARUDA', 'GATECH', 'GATECHDVR', 'GCSL', 'GEMAROMA', 'GKENERGY', 'GLOBECIVIL', 'GLO

In [ ]:
import pandas as pd
import os
import glob

INPUT_DIR = "part3_chunks"
OUTPUT_FILE = "nse_lstm_windows.parquet"
REPORT_FILE = "part3_merge_report.csv"

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))

if not files:
    raise FileNotFoundError(f"No parquet files found in {INPUT_DIR}")

dfs = []
report_rows = []

for f in files:
    df = pd.read_parquet(f)
    dfs.append(df)
    report_rows.append({
        "file": os.path.basename(f),
        "rows": len(df),
        "cols": len(df.columns)
    })

merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_parquet(OUTPUT_FILE, index=False)

report_df = pd.DataFrame(report_rows)
report_df.to_csv(REPORT_FILE, index=False)

summary = pd.DataFrame({
    "metric": ["files_merged", "total_rows", "total_cols"],
    "value": [len(files), len(merged_df), len(merged_df.columns)]
})
summary.to_csv("part3_merge_summary.csv", index=False)

print("Saved:", OUTPUT_FILE)
print("Saved:", REPORT_FILE)
print("Saved: part3_merge_summary.csv")
print(summary)

Saved: nse_lstm_windows.parquet
Saved: part3_merge_report.csv
Saved: part3_merge_summary.csv
         metric  value
0  files_merged    209
1    total_rows  48198
2    total_cols   1065


In [ ]:
import pandas as pd
import os
import glob

INPUT_DIR = "part3_chunks"
OUTPUT_FILE = "nse_lstm_windows.parquet"
REPORT_FILE = "part3_merge_report.csv"

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))

if not files:
    raise FileNotFoundError(f"No parquet files found in {INPUT_DIR}")

dfs = []
report_rows = []

for f in files:
    df = pd.read_parquet(f)
    dfs.append(df)
    report_rows.append({
        "file": os.path.basename(f),
        "rows": len(df),
        "cols": len(df.columns)
    })

merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_parquet(OUTPUT_FILE, index=False)

report_df = pd.DataFrame(report_rows)
report_df.to_csv(REPORT_FILE, index=False)

summary = pd.DataFrame({
    "metric": ["files_merged", "total_rows", "total_cols"],
    "value": [len(files), len(merged_df), len(merged_df.columns)]
})
summary.to_csv("part3_merge_summary.csv", index=False)

print("Saved:", OUTPUT_FILE)
print("Saved:", REPORT_FILE)
print("Saved: part3_merge_summary.csv")
print(summary)

Saved: nse_lstm_windows.parquet
Saved: part3_merge_report.csv
Saved: part3_merge_summary.csv
         metric  value
0  files_merged    209
1    total_rows  48198
2    total_cols   1065


!!!!!!!!!!! ***Model alert*** !!!!!!!!!!!!!!!!

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

TEST_MODE = False
TEST_SYMBOLS = 5
TEST_EPOCHS = 3
TEST_BATCH_SIZE = 64

INPUT_FILE = "nse_lstm_windows_filtered.parquet"
MODEL_FILE = "lstm_stock_model_dense5_test.keras" if TEST_MODE else "lstm_stock_model_dense5.keras"
SCALER_FILE = "target_scaler_filtered_test.pkl" if TEST_MODE else "target_scaler_filtered.pkl"
RESULTS_FILE = "part4_results_test.csv" if TEST_MODE else "part4_results.csv"
FORECAST_FILE = "part4_next5_forecast_test.csv" if TEST_MODE else "part4_next5_forecast.csv"

LOOKBACK = 30
HORIZON = 5
MASK_VALUE = -99.0
TRAIN_END = pd.Timestamp("2025-06-30")
VAL_START = pd.Timestamp("2025-07-01")
VAL_END = pd.Timestamp("2025-12-31")

df = pd.read_parquet(INPUT_FILE)
if "next_date" not in df.columns: df["next_date"] = pd.NaT
if "next_date" not in df.columns: df["next_date"] = pd.NaT
df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], errors="coerce")
df["next_date"] = pd.to_datetime(df["next_date"], errors="coerce")
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df = df.loc[:, ~df.columns.duplicated()].copy()

if TEST_MODE:
    keep_symbols = df["symbol"].dropna().drop_duplicates().tolist()[:TEST_SYMBOLS]
    df = df[df["symbol"].isin(keep_symbols)].copy()
    print("TEST MODE symbols:", keep_symbols)

base_features = [
    "Open", "High", "Low", "Close", "Volume",
    "log_return", "rsi_14", "ema_12", "ema_26", "macd", "macd_signal",
    "sma_20", "sma_50", "dist_sma_20", "dist_sma_50",
    "roc_3", "roc_5", "roc_10", "bb_width", "atr_14", "atr_pct",
    "hist_vol_10", "hist_vol_21", "vol_zscore", "obv", "mfi_14",
    "body_size", "upper_shadow_ratio", "lower_shadow_ratio",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "bench_corr_20", "rel_strength"
]
base_features = sorted([c for c in base_features if f"{c}_t1" in df.columns])

target_cols = [f"y_t{i}" for i in range(1, HORIZON + 1)]
if not all(c in df.columns for c in target_cols):
    raise ValueError("Missing multi-step target columns. Rebuild Part 3 to include y_t1...y_t5.")

df = df.sort_values(["window_end", "symbol"]).reset_index(drop=True)

train_df = df[df["window_end"] <= TRAIN_END].copy()
val_df = df[(df["window_end"] >= VAL_START) & (df["window_end"] <= VAL_END)].copy()

for c in target_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
    val_df[c] = pd.to_numeric(val_df[c], errors="coerce")

def make_xy(frame):
    X = np.zeros((len(frame), LOOKBACK, len(base_features)), dtype=np.float32)
    for t in range(LOOKBACK):
        cols = [f"{c}_t{t+1}" for c in base_features]
        X[:, t, :] = frame[cols].to_numpy(dtype=np.float32, copy=False)
    y = frame[target_cols].to_numpy(dtype=np.float32, copy=False)
    return X, y

X_train, y_train = make_xy(train_df)
X_val, y_val = make_xy(val_df)

train_valid = ~np.isnan(y_train).any(axis=1)
val_valid = ~np.isnan(y_val).any(axis=1)

X_train, y_train = X_train[train_valid], y_train[train_valid]
X_val, y_val = X_val[val_valid], y_val[val_valid]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("Sample y_train[0]:", y_train[0] if len(y_train) else "EMPTY")

target_scaler = joblib.load(SCALER_FILE)
y_train_scaled = y_train
y_val_scaled = y_val

tf.random.set_seed(42)
np.random.seed(42)

import tensorflow.keras.backend as K
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Attention, GlobalAveragePooling1D
from tensorflow.keras import Model

@tf.keras.utils.register_keras_serializable()
def asymmetric_mse(y_true, y_pred):
    sq_err = K.square(y_pred - y_true)
    penalty = K.cast(K.less(y_pred, 0.0) & K.greater(y_true, 0.0), 'float32') * 2.0
    return K.mean(sq_err * (1.0 + penalty))

inputs = Input(shape=(LOOKBACK, len(base_features)))
x = LSTM(64, return_sequences=True,
         kernel_regularizer=regularizers.l2(1e-4),
         recurrent_regularizer=regularizers.l2(1e-4))(inputs)
x = Dropout(0.25)(x)
x = Attention()([x, x])
x = GlobalAveragePooling1D()(x)
outputs = Dense(HORIZON, activation="linear", kernel_initializer="he_normal")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse"
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=5, factor=0.5, min_lr=1e-5),
    keras.callbacks.ModelCheckpoint(MODEL_FILE, monitor="val_loss", save_best_only=True)
]

history = model.fit(
    X_train, y_train_scaled,
    validation_data=(X_val, y_val_scaled),
    epochs=100 if not TEST_MODE else TEST_EPOCHS,
    batch_size=128 if not TEST_MODE else TEST_BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

model = keras.models.load_model(MODEL_FILE)
pred_scaled = model.predict(X_val, verbose=0)
pred = target_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)
true = target_scaler.inverse_transform(y_val.reshape(-1, 1)).reshape(y_val.shape)

rmse_per_h = [np.sqrt(mean_squared_error(true[:, i], pred[:, i])) for i in range(HORIZON)]
mape_per_h = [mean_absolute_percentage_error(np.clip(np.abs(true[:, i]), 1e-8, None), np.abs(pred[:, i])) for i in range(HORIZON)]
directional_accuracy_per_h = [np.mean(np.sign(true[:, i]) == np.sign(pred[:, i])) for i in range(HORIZON)]

results = pd.DataFrame({
    "horizon": np.arange(1, HORIZON + 1),
    "rmse": rmse_per_h,
    "mape": mape_per_h,
    "directional_accuracy": directional_accuracy_per_h
})
results.to_csv(RESULTS_FILE, index=False)

last_window = df.sort_values(["window_end", "symbol"]).dropna(subset=["window_end"]).iloc[-1]
start_date = last_window["next_date"]
if pd.isna(start_date):
    start_date = last_window["window_end"]
if pd.isna(start_date):
    start_date = df["window_end"].dropna().max()

forecast_dates = pd.bdate_range(start=start_date, periods=HORIZON)

last_X = np.zeros((1, LOOKBACK, len(base_features)), dtype=np.float32)
for t in range(LOOKBACK):
    for j, c in enumerate(base_features):
        last_X[0, t, j] = last_window[f"{c}_t{t+1}"]

forecast_scaled = model.predict(last_X, verbose=0).reshape(1, -1)
forecast_vals = target_scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).reshape(1, -1).ravel()

forecast_df = pd.DataFrame({
    "forecast_date": forecast_dates,
    "step_ahead": np.arange(1, HORIZON + 1),
    "predicted_y_target": forecast_vals
})
forecast_df.to_csv(FORECAST_FILE, index=False)

print(results)
print(forecast_df)
print("Saved:", MODEL_FILE)
print("Saved:", RESULTS_FILE)
print("Saved:", FORECAST_FILE)

X_train: (14054, 30, 35) y_train: (14054, 5)
X_val: (13594, 30, 35) y_val: (13594, 5)
Sample y_train[0]: [-0.87484884 -0.87572193 -0.8424219  -0.8426488  -0.5737357 ]
Epoch 1/100

  1/110 ━━━━━━━━━━━━━━━━━━━━ 9:26 5s/step - loss: 1.1062
  2/110 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - loss: 1.1269
  4/110 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - loss: 1.0693
  6/110 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - loss: 1.0296
  8/110 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - loss: 1.0697
 10/110 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - loss: 1.0274
 12/110 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 1.0289
 14/110 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - loss: 1.0127
 16/110 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 0.9895
 18/110 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 0.9749
 20/110 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 0.9574
 22/110 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 0.9358
 24/110 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 0.9189
 26/110 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 0.9008
 28/110 ━━━━━━━━━━━━━━

**Testing #####**

In [2]:
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error, confusion_matrix

INPUT_FILE = "nse_lstm_windows_filtered.parquet"
MODEL_FILE = "lstm_stock_model_dense5.keras"
SCALER_FILE = "target_scaler_filtered.pkl"

RESULTS_FILE = "final_test_metrics_2025H2.csv"
PRED_FILE = "final_test_predictions_2025H2.csv"
REPORT_FILE = "final_test_report_2025H2.csv"
SCALER_REPORT_FILE = "final_test_scaler_report.csv"

LOOKBACK = 30
HORIZON = 5
TEST_START = pd.Timestamp("2025-07-01")
TEST_END = pd.Timestamp("2025-12-31")

df = pd.read_parquet(INPUT_FILE)
if "next_date" not in df.columns: df["next_date"] = pd.NaT
if "next_date" not in df.columns: df["next_date"] = pd.NaT
df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], errors="coerce")
df["next_date"] = pd.to_datetime(df["next_date"], errors="coerce")
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df = df.loc[:, ~df.columns.duplicated()].copy()

base_features = [
    "Open", "High", "Low", "Close", "Volume",
    "log_return", "rsi_14", "ema_12", "ema_26", "macd", "macd_signal",
    "sma_20", "sma_50", "dist_sma_20", "dist_sma_50",
    "roc_3", "roc_5", "roc_10", "bb_width", "atr_14", "atr_pct",
    "hist_vol_10", "hist_vol_21", "vol_zscore", "obv", "mfi_14",
    "body_size", "upper_shadow_ratio", "lower_shadow_ratio",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "bench_corr_20", "rel_strength"
]
base_features = sorted([c for c in base_features if f"{c}_t1" in df.columns])

target_cols = [f"y_t{i}" for i in range(1, HORIZON + 1)]
if not all(c in df.columns for c in target_cols):
    raise ValueError("Missing target columns y_t1...y_t5")

test_df = df[(df["window_end"] >= TEST_START) & (df["window_end"] <= TEST_END)].copy()
test_df = test_df.sort_values(["window_end", "symbol"]).reset_index(drop=True)

for c in target_cols:
    test_df[c] = pd.to_numeric(test_df[c], errors="coerce")

def make_xy(frame):
    X = np.zeros((len(frame), LOOKBACK, len(base_features)), dtype=np.float32)
    for t in range(LOOKBACK):
        cols = [f"{c}_t{t+1}" for c in base_features]
        X[:, t, :] = frame[cols].to_numpy(dtype=np.float32, copy=False)
    y = frame[target_cols].to_numpy(dtype=np.float32, copy=False)
    return X, y

X_test, y_test = make_xy(test_df)
valid_mask = ~np.isnan(y_test).any(axis=1)
X_test, y_test = X_test[valid_mask], y_test[valid_mask]
test_df = test_df.loc[valid_mask].copy()

model = tf.keras.models.load_model(MODEL_FILE)
scaler = joblib.load(SCALER_FILE)

pred_scaled = model.predict(X_test, verbose=0)
pred = scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)
true = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)

metrics_rows = []
for i in range(HORIZON):
    rmse = np.sqrt(mean_squared_error(true[:, i], pred[:, i]))
    mae = mean_absolute_error(true[:, i], pred[:, i])
    mape = mean_absolute_percentage_error(np.clip(np.abs(true[:, i]), 1e-8, None), np.abs(pred[:, i]))
    da = np.mean(np.sign(true[:, i]) == np.sign(pred[:, i]))

    y_true_up = (true[:, i] > 0).astype(int)
    y_pred_up = (pred[:, i] > 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_up, y_pred_up, labels=[0, 1]).ravel()

    naive_pred = test_df[target_cols[i]].shift(1).fillna(0).to_numpy()
    naive_rmse = np.sqrt(mean_squared_error(true[:, i], naive_pred))
    naive_mae = mean_absolute_error(true[:, i], naive_pred)

    metrics_rows.append({
        "horizon": i + 1,
        "rmse": rmse,
        "mae": mae,
        "mape": mape,
        "directional_accuracy": da,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "naive_rmse": naive_rmse,
        "naive_mae": naive_mae
    })

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(RESULTS_FILE, index=False)

pred_df = test_df[["symbol", "window_end", "target_start", "target_end"]].copy()
for i in range(HORIZON):
    pred_df[f"true_y_t{i+1}"] = true[:, i]
    pred_df[f"pred_y_t{i+1}"] = pred[:, i]
    pred_df[f"abs_err_t{i+1}"] = np.abs(true[:, i] - pred[:, i])
    pred_df[f"dir_true_t{i+1}"] = (true[:, i] > 0).astype(int)
    pred_df[f"dir_pred_t{i+1}"] = (pred[:, i] > 0).astype(int)

pred_df.to_csv(PRED_FILE, index=False)

sample_cols = ["symbol", "window_end"] + [f"true_y_t{i}" for i in range(1, HORIZON + 1)] + [f"pred_y_t{i}" for i in range(1, HORIZON + 1)]
sample_df = pred_df[sample_cols].copy()
sample_df.to_csv(REPORT_FILE, index=False)

scaler_report = pd.DataFrame([{
    "scaler_type": type(scaler).__name__,
    "data_min": getattr(scaler, "data_min_", [np.nan])[0] if hasattr(scaler, "data_min_") else np.nan,
    "data_max": getattr(scaler, "data_max_", [np.nan])[0] if hasattr(scaler, "data_max_") else np.nan,
    "mean": getattr(scaler, "mean_", [np.nan])[0] if hasattr(scaler, "mean_") else np.nan,
    "scale": getattr(scaler, "scale_", [np.nan])[0] if hasattr(scaler, "scale_") else np.nan
}])
scaler_report.to_csv(SCALER_REPORT_FILE, index=False)

print("Metrics:")
print(metrics_df)
print("\nScaler:")
print(scaler_report)
print("\nSample predictions:")
print(sample_df.head())
print("Saved:", RESULTS_FILE)
print("Saved:", PRED_FILE)
print("Saved:", REPORT_FILE)
print("Saved:", SCALER_REPORT_FILE)

Metrics:
   horizon      rmse       mae      mape  ...    fn    tp  naive_rmse  naive_mae
0        1  0.268279  0.192820  4.934598  ...  2343  1948    0.832390   0.638004
1        2  0.269394  0.193416  5.222362  ...  2357  1930    0.831773   0.637215
2        3  0.269400  0.193170  5.087471  ...  2369  1899    0.829139   0.635869
3        4  0.269039  0.192619  5.084646  ...  2456  1809    0.827754   0.635125
4        5  0.268038  0.192708  5.089951  ...  2284  2015    0.827987   0.635475

[5 rows x 11 columns]

Scaler:
      scaler_type  data_min  data_max      mean     scale
0  StandardScaler       NaN       NaN -0.178857  0.422924

Sample predictions:
       symbol window_end  true_y_t1  ...  pred_y_t3  pred_y_t4  pred_y_t5
0   AADHARHFC 2025-07-01  -0.092649  ...  -0.185970  -0.190839  -0.166745
1     ABINFRA 2025-07-01  -0.334563  ...  -0.408029  -0.389657  -0.406406
2   ACMESOLAR 2025-07-01   0.071988  ...   0.020538  -0.008159   0.029689
3      AFCONS 2025-07-01  -0.034684  ...

debug

In [3]:
import joblib
import numpy as np
import pandas as pd

INPUT_FILE = "nse_lstm_windows_filtered.parquet"
SCALER_FILE = "target_scaler_filtered.pkl"
TRAIN_END = pd.Timestamp("2025-06-30")
TEST_START = pd.Timestamp("2025-07-01")
TEST_END = pd.Timestamp("2025-12-31")
HORIZON = 5

df = pd.read_parquet(INPUT_FILE)
if "next_date" not in df.columns: df["next_date"] = pd.NaT
if "next_date" not in df.columns: df["next_date"] = pd.NaT
df["window_end"] = pd.to_datetime(df["window_end"], errors="coerce")
df = df.loc[:, ~df.columns.duplicated()].copy()

target_cols = [f"y_t{i}" for i in range(1, HORIZON + 1)]
missing = [c for c in target_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing target columns: {missing}")

train_df = df[df["window_end"] <= TRAIN_END].copy()
test_df = df[(df["window_end"] >= TEST_START) & (df["window_end"] <= TEST_END)].copy()

for c in target_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
    test_df[c] = pd.to_numeric(test_df[c], errors="coerce")

scaler = joblib.load(SCALER_FILE)

rows = []
for i, c in enumerate(target_cols, start=1):
    tr = train_df[c].dropna().to_numpy()
    te = test_df[c].dropna().to_numpy()

    rows.append({
        "horizon": i,
        "train_count": len(tr),
        "test_count": len(te),
        "train_mean": float(np.mean(tr)) if len(tr) else np.nan,
        "train_std": float(np.std(tr)) if len(tr) else np.nan,
        "train_min": float(np.min(tr)) if len(tr) else np.nan,
        "train_max": float(np.max(tr)) if len(tr) else np.nan,
        "train_pos_pct": float((tr > 0).mean()) if len(tr) else np.nan,
        "train_neg_pct": float((tr < 0).mean()) if len(tr) else np.nan,
        "test_mean": float(np.mean(te)) if len(te) else np.nan,
        "test_std": float(np.std(te)) if len(te) else np.nan,
        "test_min": float(np.min(te)) if len(te) else np.nan,
        "test_max": float(np.max(te)) if len(te) else np.nan,
        "test_pos_pct": float((te > 0).mean()) if len(te) else np.nan,
        "test_neg_pct": float((te < 0).mean()) if len(te) else np.nan,
    })

diag_df = pd.DataFrame(rows)
diag_df.to_csv("target_diagnostics.csv", index=False)

scaler_row = {
    "scaler_type": type(scaler).__name__,
    "has_mean_": hasattr(scaler, "mean_"),
    "has_scale_": hasattr(scaler, "scale_"),
    "mean_": float(scaler.mean_[0]) if hasattr(scaler, "mean_") else np.nan,
    "scale_": float(scaler.scale_[0]) if hasattr(scaler, "scale_") else np.nan,
    "data_min_": float(scaler.data_min_[0]) if hasattr(scaler, "data_min_") else np.nan,
    "data_max_": float(scaler.data_max_[0]) if hasattr(scaler, "data_max_") else np.nan,
}
pd.DataFrame([scaler_row]).to_csv("scaler_diagnostics.csv", index=False)

print(diag_df)
print(pd.DataFrame([scaler_row]))
print("Saved: target_diagnostics.csv")
print("Saved: scaler_diagnostics.csv")

   horizon  train_count  test_count  ...  test_max  test_pos_pct  test_neg_pct
0        1        14054       13594  ...  2.787398      0.537958      0.462042
1        2        14054       13594  ...  2.787398      0.538105      0.461895
2        3        14054       13594  ...  2.787398      0.537222      0.462778
3        4        14054       13594  ...  2.787398      0.537737      0.462263
4        5        14054       13594  ...  2.787398      0.540459      0.459541

[5 rows x 15 columns]
      scaler_type  has_mean_  has_scale_  ...    scale_  data_min_  data_max_
0  StandardScaler       True        True  ...  0.422924        NaN        NaN

[1 rows x 7 columns]
Saved: target_diagnostics.csv
Saved: scaler_diagnostics.csv



part 3 updated

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

INPUT_FILE = "nse_part2_features_masked_clustered.parquet"
OUTPUT_DIR = "part3_chunks"
LOOKBACK = 30
HORIZON = 5
TARGET_COL = "log_return"
MAX_STOCKS = None   # set None later for full run

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_parquet(INPUT_FILE)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df = df.loc[:, ~df.columns.duplicated()].copy()

symbols = df["symbol"].dropna().drop_duplicates().tolist()
if MAX_STOCKS is not None:
    symbols = symbols[:MAX_STOCKS]

base_features = [
    "Open", "High", "Low", "Close", "Volume",
    "log_return", "rsi_14", "ema_12", "ema_26", "macd", "macd_signal",
    "sma_20", "sma_50", "dist_sma_20", "dist_sma_50",
    "roc_3", "roc_5", "roc_10", "bb_width", "atr_14", "atr_pct",
    "hist_vol_10", "hist_vol_21", "vol_zscore", "obv", "mfi_14",
    "body_size", "upper_shadow_ratio", "lower_shadow_ratio",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "bench_corr_20", "rel_strength"
]
base_features = sorted([c for c in base_features if c in df.columns])

print("Symbols used:", symbols)
print("Features used:", base_features)

for sym in symbols:
    cols_needed = ["symbol", "start_date", "date", TARGET_COL] + base_features
    cols_needed = [c for c in cols_needed if c in df.columns]

    g = df.loc[df["symbol"] == sym, cols_needed].copy()
    g = g.sort_values("date").reset_index(drop=True)
    g = g.loc[:, ~g.columns.duplicated()].copy()

    if len(g) < LOOKBACK + HORIZON:
        print("Skipping", sym, "not enough rows")
        continue

    for c in base_features + [TARGET_COL]:
        g[c] = pd.to_numeric(g[c], errors="coerce")

    g = g.dropna(subset=base_features + [TARGET_COL]).reset_index(drop=True)

    X = g[base_features].to_numpy(dtype=np.float32, copy=False)
    y = g[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    dates = g["date"].to_numpy()
    start_dates = g["start_date"].to_numpy()

    rows = []
    last_start = len(g) - LOOKBACK - HORIZON + 1
    for start_idx in range(last_start):
        end_idx = start_idx + LOOKBACK - 1
        target_start = end_idx + 1
        target_end = target_start + HORIZON

        y_vec = y[target_start:target_end]
        if len(y_vec) < HORIZON:
            continue

        row = {
            "symbol": sym,
            "start_date": start_dates[start_idx],
            "window_start": dates[start_idx],
            "window_end": dates[end_idx],
            "target_start": dates[target_start],
            "target_end": dates[target_end - 1],
        }

        xw = X[start_idx:end_idx + 1]
        for t in range(LOOKBACK):
            for j, col in enumerate(base_features):
                row[f"{col}_t{t+1}"] = xw[t, j]

        for h in range(HORIZON):
            row[f"y_t{h+1}"] = y_vec[h]

        rows.append(row)

    if rows:
        out = pd.DataFrame(rows)
        out.to_parquet(os.path.join(OUTPUT_DIR, f"{sym}_windows.parquet"), index=False)
        print("Saved:", sym, out.shape)

    del g, X, y, dates, start_dates, rows
    gc.collect()

print("Done. Saved chunk files to:", OUTPUT_DIR)

Symbols used: ['AADHARHFC', 'ABCOTS', 'ABINFRA', 'ACMESOLAR', 'ACUTAAS', 'ADVANCE', 'AEGISVOPAK', 'AEROENTER', 'AFCONS', 'AFFORDABLE', 'AGARWALEYE', 'AGIIL', 'AHCL', 'AJAXENGG', 'ALIVUS', 'ALLDIGI', 'ALLTIME', 'AMANTA', 'ANTELOPUS', 'ANTHEM', 'ANUHPHR', 'AQYLON', 'ARFIN', 'ARKADE', 'ARSSBL', 'ASKAUTOLTD', 'ATHERENERG', 'ATLANTAELE', 'AVANTEL', 'AVL', 'AWFIS', 'BANSALWIRE', 'BCG', 'BELLACASA', 'BELRISE', 'BHARATSE', 'BIRLANU', 'BLACKBUCK', 'BLUECOAST', 'BLUEJET', 'BLUSPRING', 'BMWVENTLTD', 'BORANA', 'CANHLIFE', 'CARRARO', 'CEMPRO', 'CEWATER', 'CHEMBONDCH', 'CIFL', 'COHANCE', 'COMSYN', 'CPCAP', 'CPEDU', 'CPPLUS', 'CRAMC', 'CRIZAC', 'DAMCAPITAL', 'DBEIL', 'DDEVPLSTIK', 'DENTA', 'DEVX', 'DIFFNKG', 'EASTSILK', 'EBGNG', 'ECOSMOBLTY', 'EFCIL', 'EIEL', 'ELLEN', 'EMBDL', 'EPACKPEB', 'ETERNAL', 'EUREKAFORB', 'EUROBOND', 'EUROPRATIK', 'FABTECH', 'FILATFASH', 'FINKURVE', 'FISCHER', 'FLAIR', 'GANESHCP', 'GANESHHOU', 'GARUDA', 'GATECH', 'GATECHDVR', 'GCSL', 'GEMAROMA', 'GKENERGY', 'GLOBECIVIL', 'GLO

In [ ]:
import pandas as pd
import os
import glob

INPUT_DIR = "part3_chunks"
OUTPUT_FILE = "nse_lstm_windows.parquet"
REPORT_FILE = "part3_merge_report.csv"

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))

if not files:
    raise FileNotFoundError(f"No parquet files found in {INPUT_DIR}")

dfs = []
report_rows = []

for f in files:
    df = pd.read_parquet(f)
    dfs.append(df)
    report_rows.append({
        "file": os.path.basename(f),
        "rows": len(df),
        "cols": len(df.columns)
    })

merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_parquet(OUTPUT_FILE, index=False)

report_df = pd.DataFrame(report_rows)
report_df.to_csv(REPORT_FILE, index=False)

summary = pd.DataFrame({
    "metric": ["files_merged", "total_rows", "total_cols"],
    "value": [len(files), len(merged_df), len(merged_df.columns)]
})
summary.to_csv("part3_merge_summary.csv", index=False)

print("Saved:", OUTPUT_FILE)
print("Saved:", REPORT_FILE)
print("Saved: part3_merge_summary.csv")
print(summary)

Saved: nse_lstm_windows.parquet
Saved: part3_merge_report.csv
Saved: part3_merge_summary.csv
         metric  value
0  files_merged    204
1    total_rows  48191
2    total_cols   1061


part 4 updated

In [1]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow


In [3]:
pip install pandas  numpy tensorflow  scikitlearn

  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.26.0 Requires-Python >=3.9,<3.13; 1.26.1 Requires-Python >=3.9,<3.13
ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow


In [4]:
import os
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, confusion_matrix

INPUT_FILE = "nse_lstm_windows_filtered.parquet"
MODEL_FILE = "lstm_stock_model_dense5.keras"
SCALER_FILE = "target_scaler_filtered.pkl"
RESULTS_FILE = "part4_results.csv"
PRED_FILE = "part4_predictions.csv"
FORECAST_FILE = "part4_next5_forecast.csv"

LOOKBACK = 30
HORIZON = 5
TRAIN_END = pd.Timestamp("2025-06-30")
TEST_START = pd.Timestamp("2025-07-01")
TEST_END = pd.Timestamp("2025-12-31")

df = pd.read_parquet(INPUT_FILE)
if "next_date" not in df.columns: df["next_date"] = pd.NaT
if "next_date" not in df.columns: df["next_date"] = pd.NaT
df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], errors="coerce")
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df = df.loc[:, ~df.columns.duplicated()].copy()

base_features = [
    "Open", "High", "Low", "Close", "Volume",
    "log_return", "rsi_14", "ema_12", "ema_26", "macd", "macd_signal",
    "sma_20", "sma_50", "dist_sma_20", "dist_sma_50",
    "roc_3", "roc_5", "roc_10", "bb_width", "atr_14", "atr_pct",
    "hist_vol_10", "hist_vol_21", "vol_zscore", "obv", "mfi_14",
    "body_size", "upper_shadow_ratio", "lower_shadow_ratio",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "bench_corr_20", "rel_strength"
]
base_features = sorted([c for c in base_features if f"{c}_t1" in df.columns])

target_cols = [f"y_t{i}" for i in range(1, HORIZON + 1)]
missing = [c for c in target_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing target columns: {missing}")

df = df.sort_values(["window_end", "symbol"]).reset_index(drop=True)

train_df = df[df["window_end"] <= TRAIN_END].copy()
test_df = df[(df["window_end"] >= TEST_START) & (df["window_end"] <= TEST_END)].copy()

for c in target_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
    test_df[c] = pd.to_numeric(test_df[c], errors="coerce")

def make_xy(frame):
    X = np.zeros((len(frame), LOOKBACK, len(base_features)), dtype=np.float32)
    for t in range(LOOKBACK):
        cols = [f"{c}_t{t+1}" for c in base_features]
        X[:, t, :] = frame[cols].to_numpy(dtype=np.float32, copy=False)
    y = frame[target_cols].to_numpy(dtype=np.float32, copy=False)
    return X, y

X_train, y_train = make_xy(train_df)
X_test, y_test = make_xy(test_df)

train_valid = ~np.isnan(y_train).any(axis=1)
test_valid = ~np.isnan(y_test).any(axis=1)

X_train, y_train = X_train[train_valid], y_train[train_valid]
X_test, y_test = X_test[test_valid], y_test[test_valid]
test_df = test_df.loc[test_valid].copy()

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

target_scaler = joblib.load(SCALER_FILE)
y_train_scaled = y_train
y_test_scaled = y_test

tf.random.set_seed(42)
np.random.seed(42)

import tensorflow.keras.backend as K
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Attention, GlobalAveragePooling1D
from tensorflow.keras import Model

@tf.keras.utils.register_keras_serializable()
def asymmetric_mse(y_true, y_pred):
    sq_err = K.square(y_pred - y_true)
    penalty = K.cast(K.less(y_pred, 0.0) & K.greater(y_true, 0.0), 'float32') * 2.0
    return K.mean(sq_err * (1.0 + penalty))

inputs = Input(shape=(LOOKBACK, len(base_features)))
x = LSTM(64, return_sequences=True,
         kernel_regularizer=regularizers.l2(1e-4),
         recurrent_regularizer=regularizers.l2(1e-4))(inputs)
x = Dropout(0.25)(x)
x = Attention()([x, x])
x = GlobalAveragePooling1D()(x)
outputs = Dense(HORIZON, activation="linear", kernel_initializer="he_normal")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse"
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=5, factor=0.5, min_lr=1e-5),
    keras.callbacks.ModelCheckpoint(MODEL_FILE, monitor="val_loss", save_best_only=True)
]

history = model.fit(
    X_train, y_train_scaled,
    validation_data=(X_test, y_test_scaled),
    epochs=100,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

model = keras.models.load_model(MODEL_FILE)

pred_scaled = model.predict(X_test, verbose=0)
pred = target_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)
true = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)

rows = []
for i in range(HORIZON):
    rmse = np.sqrt(mean_squared_error(true[:, i], pred[:, i]))
    mae = mean_absolute_error(true[:, i], pred[:, i])
    mape = mean_absolute_percentage_error(np.clip(np.abs(true[:, i]), 1e-8, None), np.abs(pred[:, i]))
    da = np.mean(np.sign(true[:, i]) == np.sign(pred[:, i]))

    y_true_up = (true[:, i] > 0).astype(int)
    y_pred_up = (pred[:, i] > 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_up, y_pred_up, labels=[0, 1]).ravel()

    rows.append({
        "horizon": i + 1,
        "rmse": rmse,
        "mae": mae,
        "mape": mape,
        "directional_accuracy": da,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    })

results = pd.DataFrame(rows)
results.to_csv(RESULTS_FILE, index=False)

pred_df = test_df[["symbol", "window_end", "start_date"]].copy()
for i in range(HORIZON):
    pred_df[f"true_y_t{i+1}"] = true[:, i]
    pred_df[f"pred_y_t{i+1}"] = pred[:, i]
    pred_df[f"abs_err_t{i+1}"] = np.abs(true[:, i] - pred[:, i])
    pred_df[f"dir_true_t{i+1}"] = (true[:, i] > 0).astype(int)
    pred_df[f"dir_pred_t{i+1}"] = (pred[:, i] > 0).astype(int)

pred_df.to_csv(PRED_FILE, index=False)

last_window = df.sort_values(["window_end", "symbol"]).dropna(subset=["window_end"]).iloc[-1]
start_date = last_window["window_end"] + pd.tseries.offsets.BDay(1)
forecast_dates = pd.bdate_range(start=start_date, periods=HORIZON)

last_X = np.zeros((1, LOOKBACK, len(base_features)), dtype=np.float32)
for t in range(LOOKBACK):
    for j, c in enumerate(base_features):
        last_X[0, t, j] = last_window[f"{c}_t{t+1}"]

forecast_scaled = model.predict(last_X, verbose=0).reshape(1, -1)
forecast_vals = target_scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).reshape(1, -1).ravel()

forecast_df = pd.DataFrame({
    "forecast_date": forecast_dates,
    "step_ahead": np.arange(1, HORIZON + 1),
    "predicted_y_target": forecast_vals
})
forecast_df.to_csv(FORECAST_FILE, index=False)

print(results)
print(pred_df.head())
print(forecast_df)
print("Saved:", MODEL_FILE)
print("Saved:", RESULTS_FILE)
print("Saved:", PRED_FILE)
print("Saved:", FORECAST_FILE)
print("Scaler mean:", float(target_scaler.mean_[0]))
print("Scaler scale:", float(target_scaler.scale_[0]))

X_train: (14054, 30, 35) y_train: (14054, 5)
X_test: (13594, 30, 35) y_test: (13594, 5)
Epoch 1/100

  1/110 ━━━━━━━━━━━━━━━━━━━━ 4:22 2s/step - loss: 1.0698
  3/110 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - loss: 1.0713
  6/110 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 0.9566
  9/110 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 0.9589
 11/110 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.9180
 13/110 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.8925
 15/110 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.8733
 17/110 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.8517
 20/110 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.8254
 22/110 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.8068
 24/110 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.7914
 26/110 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.7762
 28/110 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.7646
 30/110 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.7481
 32/110 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.7423
 34/110 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - 